# Lopopolo SG Model — Semantic Update Extraction

Extracts per-word **Semantic Update (SU)** values from the Sentence Gestalt model
(Lopopolo & Rabovsky, 2024) for every item in `stims_for_modeling.csv`.

SU at word position *i* = mean absolute change in the LSTM gestalt state when word *i* is processed:
$$\text{SU}_i = \frac{1}{H}\sum_h |\text{sg}_{i,h} - \text{sg}_{i-1,h}|$$

Output saved to `outputs/lopopolo_su_sgm.csv`.

## 1. Setup

In [1]:
import sys
# Install into whichever kernel/env is currently active
!{sys.executable} -m pip install openpyxl --quiet

In [2]:
import sys
import types
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

# Paths (relative to notebook location)
BASE      = Path('.')
MODEL_DIR = (BASE / 'lopopolo_model').resolve()
CKPT      = MODEL_DIR / 'models' / 'SGM=600E_1200GS_01LR_1L_10DP_Adamax_CE.ckpt'
VOC       = MODEL_DIR / 'lexica' / 'lexicon_30k.xlsx'
STIMS     = BASE / 'stims_for_modeling.csv'
OUT       = BASE / 'outputs' / 'lopopolo_su_sgm.csv'

sys.path.insert(0, str(MODEL_DIR))

# --- Stub 1: pytorch_lightning ---
# SGM.py inherits from pl.LightningModule and calls self.save_hyperparameters()
# and self.log() inside training methods. Provide no-op versions so the class
# loads cleanly; none of the training machinery is invoked during inference.
class _LightningModule(nn.Module):
    def save_hyperparameters(self, *args, **kwargs): pass
    def log(self, *args, **kwargs):                  pass

_pl = types.ModuleType('pytorch_lightning')
_pl.LightningModule = _LightningModule
sys.modules.setdefault('pytorch_lightning', _pl)

# --- Stub 2: models.dataset ---
# SGM.py imports SGMDataset (training only, requires gensim). Never used with
# load_data=False.
class SGMDataset:
    pass

_dataset_stub = types.ModuleType('models.dataset')
_dataset_stub.SGMDataset = SGMDataset
_models_stub  = types.ModuleType('models')
_models_stub.dataset = _dataset_stub
sys.modules.setdefault('models', _models_stub)
sys.modules.setdefault('models.dataset', _dataset_stub)

from SGM import SGMnet
from utils import load_vocabulary, sentence_2int

print('Imports OK')
print(f'CUDA available: {torch.cuda.is_available()}')

Imports OK
CUDA available: True


## 2. Load Model and Vocabulary

In [3]:
# Load checkpoint manually rather than via PL's load_from_checkpoint.
# PL 2.6 passes weights_only=None to torch.load, which PyTorch 2.9 treats as True,
# rejecting the legacy pickle format in this checkpoint. Loading directly with
# weights_only=False (safe: file is from the paper's official OSF release) avoids
# that issue entirely and removes the dependency on the PL loader API.
ckpt    = torch.load(str(CKPT), weights_only=False, map_location='cpu')
hparams = {**ckpt['hyper_parameters'], 'load_data': False, 'start_file': None, 'voc_size': 30002}

sgm = SGMnet(**hparams)
sgm.load_state_dict(ckpt['state_dict'])
sgm.batch_size = 1
sgm.eval()

vocabulary, inv_vocabulary = load_vocabulary(str(VOC))

print(f'Model loaded.  Gestalt layer size: {sgm.hidden_size}')
print(f'Vocabulary size: {len(vocabulary)} entries')

Model loaded.  Gestalt layer size: 1200
Vocabulary size: 30002 entries


## 3. Load Stimuli

In [5]:
stims = pd.read_csv(STIMS)
print(f'{len(stims)} rows  |  {stims["stim_lower"].nunique()} unique sentences')
stims.head(8)

2808 rows  |  1345 unique sentences


,dataset,stim_id,stim,stim_lower,critical_word
0,frank_2015,1140,I cannot tell you more,i cannot tell you more,cannot
1,frank_2015,1141,I cannot tell you more,i cannot tell you more,tell
2,frank_2015,1142,I cannot tell you more,i cannot tell you more,you
3,frank_2015,1143,I cannot tell you more,i cannot tell you more,more
4,frank_2015,1144,Helen ran to the toilet,helen ran to the toilet,ran
5,frank_2015,1145,Helen ran to the toilet,helen ran to the toilet,the
6,frank_2015,1146,Helen ran to the toilet,helen ran to the toilet,toilet
7,frank_2015,1147,Helen ran to the toilet,helen ran to the toilet,to


## 4. Encode Unique Sentences → SU Cache

Each unique sentence is processed once. The model returns gestalt states for every
token (including the prepended SOS); SU for word *i* is the mean absolute difference
between consecutive gestalt states.

In [6]:
def encode_sentence(sentence_str: str) -> list[float]:
    """Return a list of SU values, one per word in sentence_str."""
    words  = sentence_str.strip().split()
    ints   = sentence_2int(words, vocabulary)
    ints   = [30001] + ints          # prepend SOS token

    t_in   = torch.LongTensor(ints).view(1, len(ints))
    probe  = torch.zeros(1, t_in.shape[1], 332)  # dummy probe; encode() ignores it

    with torch.no_grad():
        sg, _, _ = sgm.encode([t_in, probe, probe])

    sg = sg.squeeze(0)          # [len(words)+1, hidden_size]

    sus, prev = [], sg[0]
    for curr in sg[1:]:
        sus.append(torch.mean(torch.abs(curr - prev)).item())
        prev = curr

    return sus                  # length == len(words)


unique_sents = stims['stim_lower'].unique()
su_cache     = {}

for i, sent in enumerate(unique_sents):
    su_cache[sent] = encode_sentence(sent)
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(unique_sents)} sentences processed')

print(f'\nDone — {len(su_cache)} sentences encoded.')

  100/1345 sentences processed
  200/1345 sentences processed
  300/1345 sentences processed
  400/1345 sentences processed
  500/1345 sentences processed
  600/1345 sentences processed
  700/1345 sentences processed
  800/1345 sentences processed
  900/1345 sentences processed
  1000/1345 sentences processed
  1100/1345 sentences processed
  1200/1345 sentences processed
  1300/1345 sentences processed

Done — 1345 sentences encoded.


## 5. Extract SU at Critical Word Position

In [7]:
import string

def find_word_position(words: list[str], critical_word: str) -> int | None:
    """
    Find the position of critical_word in the word list, handling three
    sources of mismatch between stim_lower and critical_word:
      1. Case:        'Adam'    -> stim_lower has 'adam'
      2. Contractions: 'don't'  -> stim_lower has 'dont' (apostrophe stripped)
      3. Punctuation: 'ship,'   -> stim_lower token has trailing comma/period
    Strategy: normalise both sides (lowercase, strip apostrophes, strip
    trailing punctuation) and match on the normalised form.
    """
    def normalise(w):
        return w.lower().replace("'", "").replace("'", "").rstrip(string.punctuation)

    needle = normalise(critical_word)
    for i, w in enumerate(words):
        if normalise(w) == needle:
            return i
    return None


records = []

for _, row in stims.iterrows():
    words = row['stim_lower'].split()
    sus   = su_cache[row['stim_lower']]

    pos = find_word_position(words, row['critical_word'])
    su  = sus[pos] if pos is not None else float('nan')

    is_oov = (row['critical_word'].lower() not in vocabulary or
              vocabulary[row['critical_word'].lower()] == 0)

    records.append({
        'stim_id':      row['stim_id'],
        'dataset':      row['dataset'],
        'critical_word': row['critical_word'],
        'su_sgm':       su,
        'is_oov':       is_oov,
    })

results = pd.DataFrame(records)

print(f'Total rows:         {len(results)}')
print(f'NaN SUs:            {results["su_sgm"].isna().sum()}')
print(f'OOV critical words: {results["is_oov"].sum()}')
results.head(8)

Total rows:         2808
NaN SUs:            0
OOV critical words: 231


,stim_id,dataset,critical_word,su_sgm,is_oov
0,1140,frank_2015,cannot,0.066912,False
1,1141,frank_2015,tell,0.128628,False
2,1142,frank_2015,you,0.056624,False
3,1143,frank_2015,more,0.037957,False
4,1144,frank_2015,ran,0.132660,False
5,1145,frank_2015,the,0.048526,False
6,1146,frank_2015,toilet,0.042063,False
7,1147,frank_2015,to,0.085413,False


## 6. Save Output

In [8]:
results.to_csv(OUT, index=False)
print(f'Saved {len(results)} rows → {OUT}')
results['su_sgm'].describe()

Saved 2808 rows → outputs/lopopolo_su_sgm.csv


count    2808.000000
mean        0.034744
std         0.028021
min         0.003393
25%         0.013190
50%         0.024452
75%         0.050350
max         0.140698
Name: su_sgm, dtype: float64

## 7. Correlation with N400 Amplitudes

In [9]:
from scipy import stats

N400_PATH = BASE / 'combined_clean_n400.csv'

# Aggregate to one row per stimulus (mean across subjects) to avoid
# pseudo-replication, matching the approach in pipeline.ipynb
n400 = (
    pd.read_csv(N400_PATH)
    .groupby(['stim_id', 'critical_word'], as_index=False)['meanAmp_z']
    .mean()
)

# Merge SU values with N400 amplitudes
merged = results.merge(n400, on=['stim_id', 'critical_word'], how='inner')

# Analysis set: drop NaN SUs and OOV items (OOV words map to UNK so their
# SU reflects context change, not the word itself)
analysis = merged.dropna(subset=['su_sgm', 'meanAmp_z'])
analysis_in_vocab = analysis[~analysis['is_oov']]

print(f'Merged rows: {len(merged)}  |  after dropping NaN+OOV: {len(analysis_in_vocab)}\n')

# Pearson r per dataset + overall
rows = []
for label, subset in [
    *[(ds, analysis_in_vocab[analysis_in_vocab['dataset'] == ds])
      for ds in analysis_in_vocab['dataset'].unique()],
    ('ALL', analysis_in_vocab),
]:
    if len(subset) < 3:
        continue
    r, p = stats.pearsonr(subset['su_sgm'], subset['meanAmp_z'])
    rows.append({'dataset': label, 'n': len(subset), 'r': r, 'p': p})

corr_df = pd.DataFrame(rows)
corr_df['p_stars'] = corr_df['p'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else 'ns'))
)
corr_df = corr_df[['dataset', 'n', 'r', 'p', 'p_stars']]
print(corr_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Also report OOV-included correlation for comparison
rows_all = []
for label, subset in [
    *[(ds, analysis[analysis['dataset'] == ds])
      for ds in analysis['dataset'].unique()],
    ('ALL', analysis),
]:
    if len(subset) < 3:
        continue
    r, p = stats.pearsonr(subset['su_sgm'], subset['meanAmp_z'])
    rows_all.append({'dataset': label, 'n': len(subset), 'r': r, 'p': p})

print('\n(Including OOV items for reference:)')
corr_df_all = pd.DataFrame(rows_all)
corr_df_all['p_stars'] = corr_df_all['p'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else 'ns'))
)
print(corr_df_all[['dataset', 'n', 'r', 'p', 'p_stars']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

Merged rows: 2808  |  after dropping NaN+OOV: 2577

       dataset    n       r      p p_stars
    frank_2015 1649 -0.1464 0.0000     ***
michaelov_2024  494 -0.1202 0.0075      **
   ryskin_2021  434 -0.0105 0.8267      ns
           ALL 2577 -0.0989 0.0000     ***

(Including OOV items for reference:)
       dataset    n       r      p p_stars
    frank_2015 1668 -0.1479 0.0000     ***
michaelov_2024  500 -0.1155 0.0097      **
   ryskin_2021  640  0.0062 0.8755      ns
           ALL 2808 -0.0883 0.0000     ***


## 8. Sanity Check

In [10]:
# Verify SUs look sensible: higher SU for low-cloze / unexpected words is expected
example_sent = stims['stim_lower'].iloc[0]
words = example_sent.split()
sus   = su_cache[example_sent]

print(f'Sentence: "{example_sent}"')
print(f'{"Word":<15} {"SU":>8}')
print('-' * 25)
for w, s in zip(words, sus):
    print(f'{w:<15} {s:>8.4f}')

Sentence: "i cannot tell you more"
Word                  SU
-------------------------
i                 0.0748
cannot            0.0669
tell              0.1286
you               0.0566
more              0.0380
